# DeepRhythm Model Master — Physiological Deepfake Detection Baseline

**Paper**: *DeepRhythm: Exposing DeepFakes with Attentional Visual Motifs (CVPR 2020)*

### Điểm nổi bật của kiến trúc
Đây là mô hình **Baseline thuần túy** (Pure DeepRhythm) sử dụng mạng 3D CNN để trích xuất tín hiệu nhịp tim rPPG từ sự thay đổi màu sắc vi mô của tĩnh mạch trên khuôn mặt (thông qua không gian màu YCrCb). Mô hình xử lý theo chuỗi thời gian (T=32 frames liên tiếp) thay vì ảnh tĩnh.

| Component | Chi tiết |
|---|---|
| Input | Chuỗi `T=32` frames liên tiếp (video clip), chuyển sang hệ màu YCrCb |
| Feature Extractor | `rPPGNetwork3D` - Mạng 3D CNN với 4 Block, bảo toàn chiều thời gian |
| Classifier | MLP Classifier dự đoán trực tiếp từ chuỗi Token sinh học (rPPG) |

### Architecture Overview
```
Video Clip (T=32) ──► YCrCb Color Space ──► rPPGNetwork3D ─────► Temporal Bio Tokens ─────► Classifier (MLP) ──► Real/Fake
                     (64x64 resolution)     (4 blocks 3D CNN)    (16 tokens x 64 dims)
```

### Notebook Sections
0. Environment Setup
1. Configuration & Dataset Preparation (MediaPipe T=32 Clips)
2. Dataset Class & DataLoader
3. Model Architecture (Pure DeepRhythm)
4. Training Configuration
5. Training Loop
6. Evaluation — FF++ Validation (Accuracy, F1, Precision, Recall, AUC)
7. Cross-Dataset Evaluation — Celeb-DF-v2
8. Ablation Study
9. Export & Save

---
## Section 0: Environment Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q mediapipe scikit-learn matplotlib seaborn tqdm scipy
!wget -q -O /tmp/blaze_face_short_range.tflite \
  https://storage.googleapis.com/mediapipe-models/face_detector/blaze_face_short_range/float16/1/blaze_face_short_range.tflite

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os, glob, random, time, warnings, shutil, gc
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as T
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision

from sklearn.metrics import (
    accuracy_score, roc_auc_score, roc_curve, auc,
    confusion_matrix, classification_report,
    f1_score, precision_score, recall_score
)
from scipy.optimize import brentq
from scipy.interpolate import interp1d
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from collections import defaultdict

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU:   {torch.cuda.get_device_name(0)}')
    print(f'VRAM:  {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Device: cuda
GPU:   NVIDIA L4
VRAM:  23.7 GB


---
## Section 1: Configuration & Dataset Preparation

Sử dụng MediaPipe để cắt chuỗi T=32 frames liên tiếp, lưu dưới dạng file `.npy`.

In [3]:
class Config:
    # ── Paths ──
    DRIVE_ROOT   = '/content/drive/MyDrive/DoAn_Nhom4'
    FF_ZIP       = f'{DRIVE_ROOT}/FaceForensics.zip'
    CELEB_ZIP    = f'{DRIVE_ROOT}/Celeb-DF-v2.zip'
    EXTRACT_DIR  = '/content/dataset'
    CLIPS_DIR    = '/content/clips'
    FF_CLIPS     = f'{CLIPS_DIR}/ffpp'
    CELEB_CLIPS  = f'{CLIPS_DIR}/celeb_df_v2'
    SAVE_DIR     = f'{DRIVE_ROOT}/weights_deeprhythm'

    # ── Clip Extraction ──
    T                 = 32    # frames per clip (chứa được > 1 chu kỳ tim)
    CLIPS_PER_VIDEO   = 3
    FACE_CONFIDENCE   = 0.7
    FACE_PAD_X        = 0.15
    FACE_PAD_Y_TOP    = 0.40
    FACE_PAD_Y_BOTTOM = 0.20
    FACE_SIZE         = 112   # face crop resolution

    # ── Model ──
    BIO_DIM     = 64
    BIO_TOKENS  = T // 2
    DROPOUT     = 0.3

    # ── Training ──
    BATCH_SIZE           = 16
    EPOCHS               = 30
    LR                   = 1e-4
    WEIGHT_DECAY         = 1e-4
    LABEL_SMOOTHING      = 0.05
    EARLY_STOP_PATIENCE  = 5
    GRAD_CLIP            = 1.0

    SEED = 42

def set_seed(seed=Config.SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()
for k, v in vars(Config).items():
    if not k.startswith('_'): print(f'  {k:<22} = {v}')

  DRIVE_ROOT             = /content/drive/MyDrive/DoAn_Nhom4
  FF_ZIP                 = /content/drive/MyDrive/DoAn_Nhom4/FaceForensics.zip
  CELEB_ZIP              = /content/drive/MyDrive/DoAn_Nhom4/Celeb-DF-v2.zip
  EXTRACT_DIR            = /content/dataset
  CLIPS_DIR              = /content/clips
  FF_CLIPS               = /content/clips/ffpp
  CELEB_CLIPS            = /content/clips/celeb_df_v2
  SAVE_DIR               = /content/drive/MyDrive/DoAn_Nhom4/weights_deeprhythm
  T                      = 32
  CLIPS_PER_VIDEO        = 3
  FACE_CONFIDENCE        = 0.7
  FACE_PAD_X             = 0.15
  FACE_PAD_Y_TOP         = 0.4
  FACE_PAD_Y_BOTTOM      = 0.2
  FACE_SIZE              = 112
  BIO_DIM                = 64
  BIO_TOKENS             = 16
  DROPOUT                = 0.3
  BATCH_SIZE             = 16
  EPOCHS                 = 30
  LR                     = 0.0001
  WEIGHT_DECAY           = 0.0001
  LABEL_SMOOTHING        = 0.05
  EARLY_STOP_PATIENCE    = 5
  GRAD_CLIP          

In [4]:
# Giải nén FF++ và Celeb-DF
os.makedirs(Config.EXTRACT_DIR, exist_ok=True)

ff_folders = glob.glob(os.path.join(Config.EXTRACT_DIR, 'FaceForensics*'))
if not ff_folders:
    print('Extracting FaceForensics++...')
    !unzip -q "{Config.FF_ZIP}" -d "{Config.EXTRACT_DIR}/"
    ff_folders = glob.glob(os.path.join(Config.EXTRACT_DIR, 'FaceForensics*'))
FF_ROOT = ff_folders[0]

# Xử lý thư mục gốc FF++
if 'original' not in os.listdir(FF_ROOT):
    if 'FaceForensics++_C23' in os.listdir(FF_ROOT):
        FF_ROOT = os.path.join(FF_ROOT, 'FaceForensics++_C23')
    else:
        for root, dirs, _ in os.walk(FF_ROOT):
            if 'original' in dirs: FF_ROOT = root; break

celeb_folders = glob.glob(os.path.join(Config.EXTRACT_DIR, 'Celeb*'))
if not celeb_folders:
    print('\nExtracting Celeb-DF-v2...')
    !unzip -q "{Config.CELEB_ZIP}" -d "{Config.EXTRACT_DIR}/"
    celeb_folders = glob.glob(os.path.join(Config.EXTRACT_DIR, 'Celeb*'))
CELEB_ROOT = celeb_folders[0]
if not any(os.path.isdir(os.path.join(CELEB_ROOT, d)) for d in ['Celeb-real', 'Celeb-synthesis']):
    for s in glob.glob(os.path.join(CELEB_ROOT, '*')):
        if os.path.isdir(s) and 'Celeb-real' in os.listdir(s): CELEB_ROOT = s; break

In [5]:
def extract_clips_from_video(video_path, output_dir, detector):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened(): return 0
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total < Config.T: cap.release(); return 0
    video_id = os.path.splitext(os.path.basename(video_path))[0]
    os.makedirs(output_dir, exist_ok=True)

    ret, first_frame = cap.read()
    if not ret: cap.release(); return 0
    rgb = cv2.cvtColor(first_frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    result = detector.detect(mp_image)
    if not result.detections: cap.release(); return 0

    bb = result.detections[0].bounding_box
    ih, iw = first_frame.shape[:2]
    x, y, w, h = bb.origin_x, bb.origin_y, bb.width, bb.height
    px, pt, pb = int(w * Config.FACE_PAD_X), int(h * Config.FACE_PAD_Y_TOP), int(h * Config.FACE_PAD_Y_BOTTOM)
    x1, y1 = max(0, x - px), max(0, y - pt)
    x2, y2 = min(iw, x + w + px), min(ih, y + h + pb)
    if (x2 - x1) < 20 or (y2 - y1) < 20: cap.release(); return 0

    max_start = total - Config.T
    clip_starts = np.linspace(0, max_start, Config.CLIPS_PER_VIDEO, dtype=int)
    saved = 0
    for clip_idx, start in enumerate(clip_starts):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(start))
        frames = []
        for _ in range(Config.T):
            ret, frame = cap.read()
            if not ret: break
            crop = frame[y1:y2, x1:x2]
            if crop.size == 0: break
            frames.append(cv2.resize(crop, (Config.FACE_SIZE, Config.FACE_SIZE), interpolation=cv2.INTER_LINEAR))
        if len(frames) == Config.T:
            np.save(os.path.join(output_dir, f'{video_id}_clip{clip_idx:02d}.npy'), np.stack(frames, axis=0))
            saved += 1
    cap.release()
    return saved

def process_video_folders(video_dirs, output_dir):
    base_options = mp_python.BaseOptions(model_asset_path='/tmp/blaze_face_short_range.tflite')
    options = mp_vision.FaceDetectorOptions(base_options=base_options, min_detection_confidence=Config.FACE_CONFIDENCE)
    detector = mp_vision.FaceDetector.create_from_options(options)

    videos = []
    for vdir in video_dirs:
        if not os.path.isdir(vdir): continue
        for ext in ['*.mp4', '*.avi', '*.mov', '*.mkv']:
            videos.extend(glob.glob(os.path.join(vdir, '**', ext), recursive=True))

    total_clips = 0
    for vp in tqdm(videos, desc=f'Extracting -> {os.path.basename(output_dir)}'):
        total_clips += extract_clips_from_video(vp, output_dir, detector)
    detector.close()
    return total_clips

ff_real_dir, ff_fake_dir = os.path.join(Config.FF_CLIPS, 'real'), os.path.join(Config.FF_CLIPS, 'fake')
celeb_real_dir, celeb_fake_dir = os.path.join(Config.CELEB_CLIPS, 'real'), os.path.join(Config.CELEB_CLIPS, 'fake')

# Trích xuất FF++
if os.path.exists(ff_real_dir) and len(glob.glob(f'{ff_real_dir}/*.npy')) > 100:
    print('FF++ clips already exist. Skipping extraction.')
else:
    process_video_folders([os.path.join(FF_ROOT, 'original')], ff_real_dir)
    for method in ['Deepfakes', 'Face2Face', 'FaceShifter', 'FaceSwap', 'NeuralTextures']:
        process_video_folders([os.path.join(FF_ROOT, method)], ff_fake_dir)

# Trích xuất Celeb
if os.path.exists(celeb_real_dir) and len(glob.glob(f'{celeb_real_dir}/*.npy')) > 100:
    print('Celeb-DF clips already exist. Skipping extraction.')
else:
    process_video_folders([os.path.join(Config.EXTRACT_DIR, 'Celeb-real'), os.path.join(Config.EXTRACT_DIR, 'YouTube-real')], celeb_real_dir)
    process_video_folders([os.path.join(Config.EXTRACT_DIR, 'Celeb-synthesis')], celeb_fake_dir)

Extracting -> real: 0it [00:00, ?it/s]

Extracting -> fake: 0it [00:00, ?it/s]

Extracting -> fake: 0it [00:00, ?it/s]

Extracting -> fake: 0it [00:00, ?it/s]

Extracting -> fake: 0it [00:00, ?it/s]

Extracting -> fake: 0it [00:00, ?it/s]

Celeb-DF clips already exist. Skipping extraction.


---
## Section 2: Dataset Class & DataLoader

DeepRhythm Dataset chỉ nạp chuỗi rPPG (bio_input) từ file .npy.

In [6]:
def collect_npy(base_dir):
    paths, labels = [], []
    for label_name, label_val in [('real', 0), ('fake', 1)]:
        folder = os.path.join(base_dir, label_name)
        if not os.path.exists(folder): continue
        for f in sorted(glob.glob(os.path.join(folder, '*.npy'))):
            paths.append(f); labels.append(label_val)
    return paths, labels

def get_video_id(npy_path):
    name = os.path.splitext(os.path.basename(npy_path))[0]
    idx  = name.rfind('_clip')
    return name[:idx] if idx != -1 else name

ff_paths, ff_labels = collect_npy(Config.FF_CLIPS)
video_ids = sorted(set(get_video_id(p) for p in ff_paths))
random.shuffle(video_ids)
split = int(len(video_ids) * 0.8)
train_vids = set(video_ids[:split])

train_paths, train_labels, val_paths, val_labels = [], [], [], []
for p, l in zip(ff_paths, ff_labels):
    if get_video_id(p) in train_vids:
        train_paths.append(p); train_labels.append(l)
    else:
        val_paths.append(p); val_labels.append(l)

celeb_paths, celeb_labels = collect_npy(Config.CELEB_CLIPS)

print(f"FF++ Train: {len(train_paths)} | Val: {len(val_paths)} | Celeb-DF Test: {len(celeb_paths)}")

import scipy.signal

def apply_evm(frames, alpha=15.0, low_freq=0.8, high_freq=2.5, fps=30):
    '''
    Eulerian Video Magnification (EVM) Approximation
    Magnifies subtle color variations related to heartbeats.
    '''
    frames_float = frames.astype(np.float32) / 255.0
    blurred = np.array([cv2.GaussianBlur(f, (5, 5), 0) for f in frames_float])
    
    nyq = 0.5 * fps
    b, a = scipy.signal.butter(1, [low_freq/nyq, high_freq/nyq], btype='band')
    filtered = scipy.signal.lfilter(b, a, blurred, axis=0)
    
    magnified = frames_float + filtered * alpha
    return np.clip(magnified, 0.0, 1.0) * 255.0

class DeepRhythmDataset(Dataset):
    BIO_SIZE = 64
    def __init__(self, npy_paths, labels, augment=False):
        self.paths = npy_paths
        self.labels = labels
        self.augment = augment

    def __len__(self): return len(self.paths)

    def _augment_consistent(self, frames_bgr):
        do_flip = random.random() > 0.5
        out = []
        for f in frames_bgr:
            if do_flip: f = cv2.flip(f, 1)
            out.append(f)
        return out

    def __getitem__(self, idx):
        clip_bgr = np.load(self.paths[idx], allow_pickle=False)
        frames = [clip_bgr[t] for t in range(Config.T)]
        if self.augment: frames = self._augment_consistent(frames)

        bio_frames = []
        for f in frames:
            ycrcb = cv2.cvtColor(f, cv2.COLOR_BGR2YCrCb)
            ycrcb = cv2.resize(ycrcb, (self.BIO_SIZE, self.BIO_SIZE), interpolation=cv2.INTER_LINEAR)
            bio_frames.append(ycrcb)

        bio_array = np.stack(bio_frames, axis=0).astype(np.float32) / 255.0
        bio_input = torch.from_numpy(bio_array).permute(0, 3, 1, 2)  # (T, 3, H, W)
        return bio_input, torch.tensor(self.labels[idx], dtype=torch.float32)


if len(train_labels) == 0:
    raise ValueError("ERROR: train_labels is empty! FaceForensics++ dataset was not loaded. Please check the extraction step!")

counts = np.bincount(train_labels)
sample_weights = [1.0 / counts[l] for l in train_labels]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

train_loader = DataLoader(DeepRhythmDataset(train_paths, train_labels, True), Config.BATCH_SIZE, sampler=sampler, num_workers=2, pin_memory=True)
val_loader   = DataLoader(DeepRhythmDataset(val_paths, val_labels, False), Config.BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(DeepRhythmDataset(celeb_paths, celeb_labels, False), Config.BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

FF++ Train: 0 | Val: 0 | Celeb-DF Test: 18861


ValueError: num_samples should be a positive integer value, but got num_samples=0

---
## Section 3: Model Architecture (DeepRhythm)

Chỉ sử dụng mạng 3D CNN (rPPGNetwork3D) trên miền YCrCb để phân tích sự gián đoạn nhịp tim sinh học của Deepfake.

In [ ]:
class SpatialAttention3D(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv3d(in_channels, 1, kernel_size=1),
            nn.Sigmoid()
        )
    def forward(self, x):
        t_pool = x.mean(dim=2, keepdim=True) # Average over temporal axis -> (B, C, 1, H, W)
        mask = self.conv(t_pool)             # Spatial mask -> (B, 1, 1, H, W)
        return x * mask

class TemporalAttention3D(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(in_channels, in_channels, kernel_size=3, padding=1),
            nn.Sigmoid()
        )
    def forward(self, x):
        s_pool = x.mean(dim=(3, 4))          # Global spatial pool -> (B, C, T)
        mask = self.conv(s_pool)             # Temporal mask -> (B, C, T)
        mask = mask.unsqueeze(-1).unsqueeze(-1) # (B, C, T, 1, 1)
        return x * mask

class DSTA_Block(nn.Module):
    ''' Dual-Spatial-Temporal Attention Block '''
    def __init__(self, in_channels):
        super().__init__()
        self.spatial = SpatialAttention3D(in_channels)
        self.temporal = TemporalAttention3D(in_channels)
    def forward(self, x):
        x = self.spatial(x)
        x = self.temporal(x)
        return x

class rPPGNetwork3D(nn.Module):
    ''' Motion-Magnified Spatial-Temporal Representation Network (MMSTR) '''
    def __init__(self, bio_dim=Config.BIO_DIM, bio_tokens=Config.BIO_TOKENS):
        super().__init__()

        self.block1 = nn.Sequential(
            nn.Conv3d(3, 32, kernel_size=(1, 3, 3), padding=(0, 1, 1)),
            nn.BatchNorm3d(32), nn.ReLU(inplace=True),
            nn.MaxPool3d(kernel_size=(1, 2, 2), stride=(1, 2, 2))
        )
        self.dsta1 = DSTA_Block(32)

        self.block2 = nn.Sequential(
            nn.Conv3d(32, 64, kernel_size=(3, 3, 3), padding=(1, 1, 1)),
            nn.BatchNorm3d(64), nn.ReLU(inplace=True),
            nn.MaxPool3d(kernel_size=(2, 2, 2), stride=(2, 2, 2))
        )
        self.dsta2 = DSTA_Block(64)

        self.block3 = nn.Sequential(
            nn.Conv3d(64, 128, kernel_size=(3, 3, 3), padding=(1, 1, 1)),
            nn.BatchNorm3d(128), nn.ReLU(inplace=True),
            nn.MaxPool3d(kernel_size=(1, 2, 2), stride=(1, 2, 2))
        )
        self.dsta3 = DSTA_Block(128)

        self.block4 = nn.Sequential(
            nn.Conv3d(128, bio_dim, kernel_size=(3, 3, 3), padding=(1, 1, 1)),
            nn.BatchNorm3d(bio_dim), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool3d((bio_tokens, 1, 1))
        )

    def forward(self, x):
        x = x.permute(0, 2, 1, 3, 4).contiguous() # (B, 3, T, H, W)

        x = self.dsta1(self.block1(x))
        x = self.dsta2(self.block2(x))
        x = self.dsta3(self.block3(x))

        feat = self.block4(x)
        feat = feat.squeeze(-1).squeeze(-1)       # (B, bio_dim, bio_tokens)
        return feat.permute(0, 2, 1).contiguous() # (B, bio_tokens, bio_dim)

class DeepRhythmBaseline(nn.Module):
    def __init__(self):
        super().__init__()
        self.rppg_net = rPPGNetwork3D()

        # Bi-LSTM for temporal sequence rhythm modeling
        self.lstm = nn.LSTM(
            input_size=Config.BIO_DIM,
            hidden_size=128,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=Config.DROPOUT
        )

        # Classifier MLPs
        self.classifier = nn.Sequential(
            nn.Linear(128 * 2, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.Dropout(Config.DROPOUT),
            nn.Linear(64, 1)
        )

    def forward(self, bio_input):
        bio_tokens = self.rppg_net(bio_input)     # (B, T, BIO_DIM)
        lstm_out, _ = self.lstm(bio_tokens)       # (B, T, 256)
        final_feat = lstm_out.mean(dim=1)         # Global average pool rhythm features
        return self.classifier(final_feat)

model = DeepRhythmBaseline().to(device)
print(f"✅ DeepRhythm Model Loaded | Params: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")

---
## Section 4: Training Configuration

In [ ]:
class LabelSmoothingBCELoss(nn.Module):
    def __init__(self, smoothing=Config.LABEL_SMOOTHING):
        super().__init__()
        self.s = smoothing
        self.bce = nn.BCEWithLogitsLoss()
    def forward(self, pred, target):
        target_smooth = target * (1 - self.s) + 0.5 * self.s
        return self.bce(pred, target_smooth)

class EarlyStopping:
    def __init__(self, patience=Config.EARLY_STOP_PATIENCE):
        self.patience = patience; self.counter = 0; self.best = None; self.stop = False
    def __call__(self, score):
        if self.best is None: self.best = score
        elif score < self.best + 1e-4:
            self.counter += 1
            if self.counter >= self.patience: self.stop = True
        else: self.best = score; self.counter = 0

criterion = LabelSmoothingBCELoss()
optimizer = optim.AdamW(model.parameters(), lr=Config.LR, weight_decay=Config.WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=Config.EPOCHS, eta_min=1e-6)
early_stopping = EarlyStopping()

---
## Section 5: Training Loop

In [ ]:
os.makedirs(Config.SAVE_DIR, exist_ok=True)
history = defaultdict(list)
best_val_auc = 0.0

def compute_eer(y_true, y_scores):
    fpr, tpr, _ = roc_curve(y_true, y_scores)
    return brentq(lambda x: 1.0 - x - interp1d(fpr, tpr)(x), 0.0, 1.0) * 100

for epoch in range(1, Config.EPOCHS + 1):
    t0 = time.time()

    # Train
    model.train()
    tr_loss, tr_corr, tr_total = 0.0, 0, 0
    for bios, labels in tqdm(train_loader, desc=f'Ep {epoch:02d} [Train]', leave=False):
        bios, labels = bios.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(bios).squeeze(1)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), Config.GRAD_CLIP)
        optimizer.step()

        tr_loss += loss.item() * bios.size(0)
        preds = (torch.sigmoid(outputs) > 0.5).float()
        tr_corr += (preds == labels).sum().item()
        tr_total += labels.size(0)

    tr_loss /= tr_total; tr_acc = tr_corr / tr_total

    # Val
    model.eval()
    va_loss, va_corr, va_total = 0.0, 0, 0
    all_labels, all_probs = [], []
    with torch.no_grad():
        for bios, labels in val_loader:
            bios, labels = bios.to(device), labels.to(device)
            outputs = model(bios).squeeze(1)
            loss = criterion(outputs, labels)

            va_loss += loss.item() * bios.size(0)
            preds = (torch.sigmoid(outputs) > 0.5).float()
            va_corr += (preds == labels).sum().item()
            va_total += labels.size(0)
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(torch.sigmoid(outputs).cpu().numpy())

    va_loss /= va_total; va_acc = va_corr / va_total
    va_auc = roc_auc_score(all_labels, all_probs)
    va_eer = compute_eer(all_labels, all_probs)

    scheduler.step()

    marker = ""
    if va_auc > best_val_auc:
        best_val_auc = va_auc
        torch.save(model.state_dict(), os.path.join(Config.SAVE_DIR, 'deeprhythm_best.pth'))
        marker = " ← 🏆 BEST"

    print(f"Epoch {epoch:02d} | TrLoss: {tr_loss:.4f} TrAcc: {tr_acc:.4f} | ValLoss: {va_loss:.4f} ValAcc: {va_acc:.4f} AUC: {va_auc:.4f} EER: {va_eer:.2f}% | {time.time()-t0:.0f}s{marker}")

    early_stopping(va_auc)
    if early_stopping.stop:
        print(f"Early stopping at epoch {epoch}")
        break

    torch.cuda.empty_cache()

---
## Section 6: Evaluation — FF++ Validation (Full Metrics)

In [ ]:
model.load_state_dict(torch.load(os.path.join(Config.SAVE_DIR, 'deeprhythm_best.pth'), map_location=device))
model.eval()

all_labels, all_probs = [], []
with torch.no_grad():
    for bios, labels in val_loader: # Có thể đổi sang test_loader nếu đã chia test split
        outputs = model(bios.to(device)).squeeze(1)
        all_labels.extend(labels.numpy())
        all_probs.extend(torch.sigmoid(outputs).cpu().numpy())

labels = np.array(all_labels)
probs = np.array(all_probs)
preds = (probs > 0.5).astype(int)

fpr, tpr, _ = roc_curve(labels, probs)
test_auc = auc(fpr, tpr)
eer = compute_eer(labels, probs) / 100.0
test_acc = accuracy_score(labels, preds)

print(f"==================================================")
print(f"📊 DEEPRHYTHM EVALUATION (FaceForensics++ C23)")
print(f"🎯 Accuracy  : {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"🎯 AUC       : {test_auc:.4f}")
print(f"🎯 EER       : {eer:.4f}")
print(f"🎯 Precision : {precision_score(labels, preds):.4f}")
print(f"🎯 Recall    : {recall_score(labels, preds):.4f}")
print(f"🎯 F1-Score  : {f1_score(labels, preds):.4f}")
print(f"==================================================")

print("\nClassification Report:")
print(classification_report(labels, preds, target_names=['Real (0)', 'Fake (1)']))

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

cm = confusion_matrix(labels, preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0], annot_kws={"size": 16})
axes[0].set_title('Confusion Matrix', fontweight='bold', fontsize=18)
axes[0].set_xticklabels(['Real', 'Fake']); axes[0].set_yticklabels(['Real', 'Fake'])

axes[1].plot(fpr, tpr, color='#d62728', lw=3, label=f'DeepRhythm (AUC = {test_auc:.4f} | EER = {eer:.4f})')
axes[1].plot([0,1],[0,1], color='navy', lw=2, linestyle='--', alpha=0.6)
axes[1].scatter([eer], [1-eer], s=200, marker='*', color='gold', edgecolor='black', zorder=5)
axes[1].set_xlabel('False Positive Rate', fontweight='bold'); axes[1].set_ylabel('True Positive Rate', fontweight='bold')
axes[1].set_title('ROC Curve', fontweight='bold', fontsize=18)
axes[1].legend(loc='lower right', frameon=True, shadow=True, fontsize=14)

plt.tight_layout()
plt.savefig(os.path.join(Config.SAVE_DIR, 'deeprhythm_evaluation.pdf'), format='pdf', bbox_inches='tight')
plt.show()

---
## Section 7: Cross-Dataset Evaluation — Celeb-DF-v2

In [ ]:
model.eval()
all_labels, all_probs = [], []
with torch.no_grad():
    for bios, labels in test_loader: # Đây chính là celeb_loader đã định nghĩa ở Section 2
        outputs = model(bios.to(device)).squeeze(1)
        all_labels.extend(labels.numpy())
        all_probs.extend(torch.sigmoid(outputs).cpu().numpy())

labels = np.array(all_labels)
probs = np.array(all_probs)
preds = (probs > 0.5).astype(int)

fpr, tpr, _ = roc_curve(labels, probs)
test_auc = auc(fpr, tpr)
eer = compute_eer(labels, probs) / 100.0

print(f"📊 CROSS-DATASET EVALUATION (Celeb-DF v2)")
print(f"🎯 Accuracy : {accuracy_score(labels, preds):.4f} | AUC: {test_auc:.4f} | EER: {eer:.4f}")
print("\nClassification Report (Celeb-DF):")
print(classification_report(labels, preds, target_names=['Real (0)', 'Fake (1)']))

---
## Section 8: Ablation Study

Nơi kiểm chứng sự phụ thuộc của mạng 3D CNN rPPG vào các phương pháp lấy mẫu temporal.

In [ ]:
print("Ready for Ablation Study.")

---
## Section 9: Export & Save

In [ ]:
print(f"✅ DeepRhythm training complete. All assets saved to {Config.SAVE_DIR}")